# Model Likelihood Scoring (MPNN and ESMIF)

**Paper**: *Decoding the physicochemical basis of taxonomy preferences in protein design models* (Dillon, Maiwald & Crook, 2025)

> **Note**: This notebook documents the scoring pipeline used to generate per-protein model likelihoods. 
> Pre-computed scores are included in `../data/Decoding_Bias_Dataset.csv`.
> Running this notebook requires GPU access and heavy dependencies (JAX, PyTorch, ColabDesign, ESM).
> It is provided for transparency and is **not required** for reproducing the analysis.


<a href="https://colab.research.google.com/github/LigandBindingDomain/Decoding_Bias/blob/main/Calculate_All_models_likelihoods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ProteinMPNN  (scored in the MPNN-family CLI section below)

The original ColabDesign `ar_mask=0` ProteinMPNN scorer has been **retired** - ProteinMPNN is now scored with the
dauparas CLI `-global_score` (checkpoint `v_48_020`) together with SolubleMPNN and the fine-tuned models, so the
whole family is one comparable score object. The cell below only sets up Google Drive / the session.

In [ ]:
# === Google Drive setup for persistent results ===
from google.colab import drive
import os
try:
    drive.mount('/content/drive')
except Exception as _e:
    print(f"Drive mount: {_e}")
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/decoding_bias_results/AF'
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"Results will save to: {DRIVE_RESULTS_DIR}")
# === End Drive setup ===

#@title Setup

import os
try:
  import colabdesign
except:
  os.system("pip -q install git+https://github.com/sokrypton/ColabDesign.git@v1.1.1")
  os.system("ln -s /usr/local/lib/python3.7/dist-packages/colabdesign colabdesign")

from colabdesign.mpnn import mk_mpnn_model, clear_mem
from colabdesign.shared.protein import pdb_to_string

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML
import pandas as pd
import tqdm.notebook
TQDM_BAR_FORMAT = '{l_bar}{bar}| {n_fmt}/{total_fmt} [elapsed: {elapsed} remaining: {remaining}]'

from google.colab import files
from google.colab import data_table
data_table.enable_dataframe_formatter()
# Import necessary libraries
import os
import json
import pandas as pd
import numpy as np
import requests
import logging
import time
from tqdm import tqdm
from colabdesign.mpnn import mk_mpnn_model, clear_mem
from pathlib import Path
from scipy.special import softmax
from typing import Optional, Tuple, Dict, Any

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Constants
AA_TO_INDEX = {
    'A': 0, 'R': 1, 'N': 2, 'D': 3, 'C': 4, 'Q': 5, 'E': 6, 'G': 7, 'H': 8, 'I': 9,
    'L': 10, 'K': 11, 'M': 12, 'F': 13, 'P': 14, 'S': 15, 'T': 16, 'W': 17, 'Y': 18, 'V': 19
}
PDB_DIR = "pdb_files"
MODEL_NAME = "v_48_020"
# === Resume-from-checkpoint helpers ===
import time as _t, glob as _g
SESSION_ID = _t.strftime("%Y%m%d_%H%M%S")

def get_completed_entries(model_dir):
    """Return set of Entry IDs already scored (read from any CSV in model_dir)."""
    if not os.path.isdir(model_dir):
        return set()
    done = set()
    for fn in _g.glob(os.path.join(model_dir, "*.csv")):
        try:
            _df = pd.read_csv(fn)
            if "Entry" in _df.columns:
                done.update(_df["Entry"].dropna().unique())
        except Exception as _e:
            print(f"  could not read {fn}: {_e}")
    return done

def csv_filter_resume(input_csv, model_dir):
    """Drop already-scored rows. Returns path to filtered CSV (or original if nothing done)."""
    done = get_completed_entries(model_dir)
    if not done:
        print(f"  Resume: no prior results in {model_dir}; running full CSV")
        return input_csv
    df = pd.read_csv(input_csv)
    before = len(df)
    df = df[~df["Entry"].isin(done)].reset_index(drop=True)
    print(f"  Resume: {len(done)} already done in {model_dir}; "
          f"{len(df)}/{before} remaining (session {SESSION_ID})")
    if len(df) == 0:
        print("  All entries already scored - nothing to do.")
        return None
    out = input_csv.replace(".csv", f"_remaining_{SESSION_ID}.csv")
    df.to_csv(out, index=False)
    return out
# === End resume helpers ===


## SolubleMPNN


In [ ]:
#@title Clone dauparas/ProteinMPNN (soluble weights bundled)
# SolubleMPNN weights only exist as PyTorch .pt files in dauparas/ProteinMPNN,
# not as colabdesign .pkl. So for this section we shell out to dauparas's CLI.
import os, subprocess, sys

PMPNN_REPO_DIR = '/content/ProteinMPNN_dauparas'
if not os.path.isdir(PMPNN_REPO_DIR):
    print('Cloning dauparas/ProteinMPNN...')
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/dauparas/ProteinMPNN.git',
                    PMPNN_REPO_DIR], check=True)
print('Repo at:', PMPNN_REPO_DIR)

PMPNN_RUN_PY    = os.path.join(PMPNN_REPO_DIR, 'protein_mpnn_run.py')
PMPNN_PARSE_PY  = os.path.join(PMPNN_REPO_DIR, 'helper_scripts',
                                'parse_multiple_chains.py')
assert os.path.exists(PMPNN_RUN_PY), PMPNN_RUN_PY
assert os.path.exists(PMPNN_PARSE_PY), PMPNN_PARSE_PY
print('CLI script: ', PMPNN_RUN_PY)
print('Parser:     ', PMPNN_PARSE_PY)


In [ ]:
#@title SolubleMPNN scorer (via dauparas CLI)

import os, tempfile, shutil, subprocess, logging, json, glob
import numpy as np
import pandas as pd
from tqdm import tqdm

class SolubleMPNNProcessor:
    """
    Score proteins with SolubleMPNN by shelling out to
    dauparas/ProteinMPNN/protein_mpnn_run.py --use_soluble_model --score_only 1.

    Output schema (Entry, sequence_score = -global_score, entropy=NaN, sequence_length, mean_confidence=NaN):
        Entry, sequence_score, entropy, sequence_length, mean_confidence
    where:
        sequence_score  = -global_score  (per-residue log-likelihood; higher = better)
        entropy         = NaN  (dauparas score_only does not export logits)
        mean_confidence = NaN  (likewise)
    The two NaN columns are kept for downstream-merge compatibility only.
    """
    def __init__(self, model_name='v_48_020', batch_size=1):
        self.model_name = model_name
        self.batch_size = batch_size
        logging.basicConfig(level=logging.INFO,
                            format='%(asctime)s %(levelname)s %(message)s')

    # ---- PDB fetch (same logic as the ColabDesign-based class) -------------
    def get_pdb(self, accession, output_dir=PDB_DIR):
        pdb_path = os.path.join(output_dir, f'AF-{accession}-F1-model_v6.pdb')
        if os.path.exists(pdb_path):
            return pdb_path, True
        os.makedirs(output_dir, exist_ok=True)
        try:
            r = requests.get(
                f'https://alphafold.ebi.ac.uk/files/AF-{accession}-F1-model_v6.pdb')
            if r.status_code == 200:
                with open(pdb_path, 'wb') as f:
                    f.write(r.content)
                return pdb_path, True
        except Exception as e:
            logging.error(f'PDB download failed for {accession}: {e}')
        return None, False

    # ---- Core: score a chunk by JSONL + score_only --------------------------
    def _score_chunk(self, accession_to_pdb):
        """accession_to_pdb: dict {Entry -> local PDB path}. Returns list of dicts."""
        if not accession_to_pdb:
            return []

        workdir = tempfile.mkdtemp(prefix='solublempnn_')
        pdb_in  = os.path.join(workdir, 'pdbs')
        out_dir = os.path.join(workdir, 'out')
        os.makedirs(pdb_in,  exist_ok=True)
        os.makedirs(out_dir, exist_ok=True)
        jsonl   = os.path.join(workdir, 'parsed.jsonl')

        # Stage PDBs into one directory with predictable filenames
        staged = {}
        for entry, src in accession_to_pdb.items():
            tag = entry  # keep simple; dauparas uses the filename stem as name
            dst = os.path.join(pdb_in, f'{tag}.pdb')
            shutil.copyfile(src, dst)
            staged[tag] = entry

        # Step 1: parse PDB dir -> JSONL
        subprocess.run([sys.executable, PMPNN_PARSE_PY,
                        '--input_path',  pdb_in,
                        '--output_path', jsonl], check=True)

        # Step 2: score-only with soluble weights
        subprocess.run([sys.executable, PMPNN_RUN_PY,
                        '--jsonl_path',   jsonl,
                        '--out_folder',   out_dir,
                        '--score_only',   '1',
                        '--use_soluble_model',
                        '--model_name',   self.model_name,
                        '--batch_size',   str(self.batch_size),
                        '--suppress_print', '1'], check=True)

        # Step 3: collect score_only .npz files (one per PDB)
        results = []
        for tag, entry in staged.items():
            npz_glob = glob.glob(os.path.join(out_dir, 'score_only',
                                              f'{tag}*.npz'))
            if not npz_glob:
                logging.warning(f'No score npz for {entry}')
                continue
            data = np.load(npz_glob[0])
            global_score = float(np.mean(data['global_score']))
            seq_len      = int(data['S'].shape[-1]) if 'S' in data else None
            results.append({
                'Entry':           entry,
                'sequence_score':  -global_score,  # match PMPNN convention
                'entropy':         np.nan,
                'sequence_length': seq_len,
                'mean_confidence': np.nan,
            })

        shutil.rmtree(workdir, ignore_errors=True)
        return results

    # ---- Public: process whole CSV ------------------------------------------
    def process_proteins_in_chunks(self, csv_file, chunk_size=500, chain='A'):
        df_in = pd.read_csv(csv_file)
        total = len(df_in)
        logging.info(f'SolubleMPNN: scoring {total} proteins in chunks of {chunk_size}')
        all_results = []
        for start in range(0, total, chunk_size):
            end = min(start + chunk_size, total)
            chunk = df_in.iloc[start:end]

            # Fetch / locate PDBs for this chunk
            acc2pdb = {}
            for _, row in tqdm(chunk.iterrows(), total=len(chunk),
                               desc=f'PDB fetch [{start}:{end}]'):
                p, ok = self.get_pdb(row['Entry'])
                if ok:
                    acc2pdb[row['Entry']] = p

            # Score this chunk in one shot
            chunk_results = self._score_chunk(acc2pdb)
            all_results.extend(chunk_results)

            # Save chunk to Drive
            chunk_df = pd.DataFrame(chunk_results)
            chunk_path = os.path.join(
                DRIVE_RESULTS_DIR, 'solublempnn',
                f'solubleMPNN_results_{SESSION_ID}_chunk_{start}-{end-1}.csv')
            os.makedirs(os.path.dirname(chunk_path), exist_ok=True)
            chunk_df.to_csv(chunk_path, index=False)
            logging.info(f'Saved {len(chunk_df)} scores -> {chunk_path}')

        final = pd.DataFrame(all_results)
        final_path = os.path.join(
            DRIVE_RESULTS_DIR, 'solublempnn',
            f'solubleMPNN_results_all_{SESSION_ID}.csv')
        os.makedirs(os.path.dirname(final_path), exist_ok=True)
        final.to_csv(final_path, index=False)
        logging.info(f'Final results -> {final_path}')
        return final


In [ ]:
#@title Run SolubleMPNN
processor = SolubleMPNNProcessor(model_name='v_48_020')

process_mode = input("Enter mode ('csv' or 'single'): ").strip().lower()
if process_mode == 'csv':
    print('Please upload a CSV file containing protein data:')
    uploaded = files.upload()
    if not uploaded:
        print('No file uploaded. Exiting.')
    else:
        csv_path = list(uploaded.keys())[0]
        chunk_size = 500  # smaller default - CLI subprocess has overhead
        csv_path = csv_filter_resume(csv_path,
                       os.path.join(DRIVE_RESULTS_DIR, 'solublempnn'))
        if csv_path is not None:
            processor.process_proteins_in_chunks(csv_path, chunk_size=chunk_size)
elif process_mode == 'single':
    accession = input('Enter AlphaFold accession (e.g. P0DTD1): ').strip()
    p, ok = processor.get_pdb(accession)
    if not ok:
        print(f'Failed to fetch PDB for {accession}')
    else:
        out = processor._score_chunk({accession: p})
        if out:
            print('SolubleMPNN Result:')
            for k, v in out[0].items():
                print(f'  {k}: {v}')
else:
    print("Invalid mode. Use 'csv' or 'single'.")


## ProteinMPNN (CLI) + AlkSecMPNN / AcidSecMPNN - the whole MPNN family in one score object

All MPNN-family scoring uses the **dauparas CLI `--score_only 1` → autoregressive `-global_score`** (the model's
native likelihood), so ProteinMPNN, SolubleMPNN and the fine-tuned models are one consistent object.
`proteinmpnn_v48_020_cli` is the **main ProteinMPNN score** (replacing the retired ColabDesign `ar_mask=0`
scoring); `proteinmpnn_v48_002_base` is included **only** as the matched base for the fine-tuned models.
Requires the SolubleMPNN section above (clones the dauparas repo, defines `SolubleMPNNProcessor`).

In [ ]:
#@title Upload fine-tuned weights  (ft_mpnn_weights.zip → ft_weights/{AlkSecMPNN,AcidSecMPNN}.pt)
import os, zipfile, torch
FT_WEIGHTS_DIR = "/content/ft_weights"
if not (os.path.isdir(FT_WEIGHTS_DIR) and any(f.endswith(".pt") for f in os.listdir(FT_WEIGHTS_DIR))):
    print("Upload finetune/colab/ft_mpnn_weights.zip:")
    up = files.upload()
    with zipfile.ZipFile(next(iter(up))) as z:
        z.extractall("/content")
# CLI reads checkpoint["noise_level"]; add it if a checkpoint lacks it (cosmetic, printed only)
for f in os.listdir(FT_WEIGHTS_DIR):
    if f.endswith(".pt"):
        fp = os.path.join(FT_WEIGHTS_DIR, f); ck = torch.load(fp, map_location="cpu")
        if "noise_level" not in ck:
            ck["noise_level"] = 0.2; torch.save(ck, fp); print("  added noise_level ->", f)
print("FT weights ready:", sorted(os.listdir(FT_WEIGHTS_DIR)))


In [ ]:
#@title Fine-tuned / base MPNN scorer (dauparas CLI - same object as SolubleMPNN)
class FinetunedMPNNProcessor(SolubleMPNNProcessor):
    """Score with a fine-tuned ProteinMPNN checkpoint (weights_dir → --path_to_model_weights) OR a
    vanilla base (weights_dir='' → repo vanilla_model_weights). Identical CLI score object & output
    schema as SolubleMPNNProcessor; only the model-selection flags and output subdir differ."""
    def __init__(self, model_name, weights_dir="", batch_size=1):
        super().__init__(model_name=model_name, batch_size=batch_size)
        self.weights_dir = weights_dir

    def _score_chunk(self, accession_to_pdb):
        if not accession_to_pdb:
            return []
        workdir = tempfile.mkdtemp(prefix="ftmpnn_")
        pdb_in = os.path.join(workdir, "pdbs"); out_dir = os.path.join(workdir, "out")
        os.makedirs(pdb_in, exist_ok=True); os.makedirs(out_dir, exist_ok=True)
        jsonl = os.path.join(workdir, "parsed.jsonl"); staged = {}
        for entry, src in accession_to_pdb.items():
            shutil.copyfile(src, os.path.join(pdb_in, f"{entry}.pdb")); staged[entry] = entry
        subprocess.run([sys.executable, PMPNN_PARSE_PY, "--input_path", pdb_in,
                        "--output_path", jsonl], check=True)
        cmd = [sys.executable, PMPNN_RUN_PY, "--jsonl_path", jsonl, "--out_folder", out_dir,
               "--score_only", "1", "--model_name", self.model_name,
               "--batch_size", str(self.batch_size), "--suppress_print", "1"]
        if self.weights_dir:                       # fine-tuned weights; else repo vanilla_model_weights
            cmd += ["--path_to_model_weights", self.weights_dir]
        subprocess.run(cmd, check=True)
        results = []
        for tag, entry in staged.items():
            g = glob.glob(os.path.join(out_dir, "score_only", f"{tag}*.npz"))
            if not g:
                logging.warning(f"No score npz for {entry}"); continue
            d = np.load(g[0])
            results.append({"Entry": entry, "sequence_score": -float(np.mean(d["global_score"])),
                            "entropy": np.nan,
                            "sequence_length": int(d["S"].shape[-1]) if "S" in d else None,
                            "mean_confidence": np.nan})
        shutil.rmtree(workdir, ignore_errors=True)
        return results

    def process_proteins_in_chunks(self, csv_file, chunk_size=500, chain="A", subdir=None):
        subdir = subdir or self.model_name.lower()
        df_in = pd.read_csv(csv_file); total = len(df_in); all_results = []
        logging.info(f"{subdir}: scoring {total} proteins in chunks of {chunk_size}")
        for start in range(0, total, chunk_size):
            end = min(start + chunk_size, total); chunk = df_in.iloc[start:end]; acc2pdb = {}
            for _, row in tqdm(chunk.iterrows(), total=len(chunk), desc=f"{subdir} PDB [{start}:{end}]"):
                pth, ok = self.get_pdb(row["Entry"])
                if ok: acc2pdb[row["Entry"]] = pth
            res = self._score_chunk(acc2pdb); all_results.extend(res)
            cp = os.path.join(DRIVE_RESULTS_DIR, subdir, f"{subdir}_results_{SESSION_ID}_chunk_{start}-{end-1}.csv")
            os.makedirs(os.path.dirname(cp), exist_ok=True); pd.DataFrame(res).to_csv(cp, index=False)
            logging.info(f"saved {len(res)} -> {cp}")
        final = pd.DataFrame(all_results)
        fp = os.path.join(DRIVE_RESULTS_DIR, subdir, f"{subdir}_results_all_{SESSION_ID}.csv")
        os.makedirs(os.path.dirname(fp), exist_ok=True); final.to_csv(fp, index=False)
        logging.info(f"final -> {fp}"); return final


In [ ]:
#@title Run ProteinMPNN (v_48_020 CLI) + matched base (v_48_002) + AlkSecMPNN + AcidSecMPNN
print("Upload the protein CSV (columns: Entry, sequence):")
uploaded = files.upload()
csv_path_ft = list(uploaded.keys())[0]

MODELS = [
    ("proteinmpnn_v48_020_cli", "v_48_020", ""),      # bridge / default ProteinMPNN (the main score)
    ("proteinmpnn_v48_002_base", "v_48_002", ""),     # matched base for the fine-tuned models only
    ("AlkSecMPNN",  "AlkSecMPNN",   FT_WEIGHTS_DIR),
    ("AcidSecMPNN","AcidSecMPNN", FT_WEIGHTS_DIR),
]
for subdir, model_name, wdir in MODELS:
    print(f"\n=== {subdir} ===")
    csvp = csv_filter_resume(csv_path_ft, os.path.join(DRIVE_RESULTS_DIR, subdir))
    if csvp is None:
        print("  already complete (resume)"); continue
    FinetunedMPNNProcessor(model_name=model_name, weights_dir=wdir
                          ).process_proteins_in_chunks(csvp, chunk_size=500, subdir=subdir)


In [ ]:
#@title Base → FT score-shift comparison  (Δscore vs pI / charge - quantify the bias as scores land)
# Compares each fine-tuned MPNN to its v_48_002 base over the full dataset, and to each other.
# Δscore = FT − base (per-residue log-likelihood; + = the FT prefers it more). AlkSecMPNN should
# raise acidic-protein likelihood (Δ falls with pI/charge); AcidSecMPNN the opposite = the polar bias.
import os, glob, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from Bio.SeqUtils.ProtParam import ProteinAnalysis

def _load(subdir):
    f = os.path.join(DRIVE_RESULTS_DIR, subdir, f"{subdir}_results_all_{SESSION_ID}.csv")
    if os.path.exists(f):
        return pd.read_csv(f)
    ch = glob.glob(os.path.join(DRIVE_RESULTS_DIR, subdir, f"{subdir}_results_*chunk*.csv"))
    return pd.concat([pd.read_csv(c) for c in ch], ignore_index=True) if ch else pd.DataFrame()

BASE = "proteinmpnn_v48_002_base"
FT_MODELS = ["AlkSecMPNN", "AcidSecMPNN"]          # whichever are present
COLORS = {"AlkSecMPNN": "#c0392b", "AcidSecMPNN": "#2980b9"}

base = _load(BASE); assert len(base), f"need {BASE} scores first"
m = base[["Entry", "sequence_score"]].rename(columns={"sequence_score": BASE})
present = []
for ft in FT_MODELS:
    d = _load(ft)
    if len(d):
        m = m.merge(d[["Entry", "sequence_score"]].rename(columns={"sequence_score": ft}), on="Entry", how="inner")
        present.append(ft)
    else:
        print(f"  (no scores for {ft} yet - skip)")
assert present, "no fine-tuned model scores found"

# per-protein properties from the uploaded CSV (Entry, sequence)
STD = set("ACDEFGHIKLMNPQRSTVWY")
def _props(s):
    c = "".join(x for x in str(s) if x in STD)
    if not c:
        return (np.nan, np.nan, np.nan)
    pa = ProteinAnalysis(c)
    return (pa.isoelectric_point(), pa.charge_at_pH(7.0) / len(c), (c.count("D") + c.count("E")) / len(c))
seqdf = pd.read_csv(csv_path_ft)[["Entry", "sequence"]].drop_duplicates("Entry")
seqdf[["pI", "charge", "acidic"]] = seqdf["sequence"].apply(lambda s: pd.Series(_props(s)))
m = m.merge(seqdf.drop(columns="sequence"), on="Entry", how="left").dropna(subset=["pI", "charge"])
for ft in present:
    m["d_" + ft] = m[ft] - m[BASE]
print(f"merged {len(m)} proteins | fine-tuned models: {', '.join(present)}\n")

# ---- summary table ----
print(f"{'model':<16}{'mean Δ':>9}{'corr(Δ,charge)':>16}{'corr(Δ,pI)':>12}{'corr(Δ,acidic)':>15}")
for ft in present:
    s = m[["d_" + ft, "charge", "pI", "acidic"]].dropna()
    print(f"{ft:<16}{m['d_'+ft].mean():>9.3f}{pearsonr(s['d_'+ft],s.charge)[0]:>16.3f}"
          f"{pearsonr(s['d_'+ft],s.pI)[0]:>12.3f}{pearsonr(s['d_'+ft],s.acidic)[0]:>15.3f}")
print("\nAlkSecMPNN: corr(Δ,charge/pI) < 0 = prefers acidic.  AcidSecMPNN: > 0 = prefers basic. Opposite signs = polar.")

# ---- Figure 1: Δscore vs net charge and vs pI, per fine-tuned model ----
fig, axes = plt.subplots(len(present), 2, figsize=(11, 4.2 * len(present)), squeeze=False)
for i, ft in enumerate(present):
    for j, (xcol, xlab) in enumerate([("charge", "native net charge / residue (pH 7)"), ("pI", "isoelectric point  pI")]):
        a = axes[i, j]; s = m[["d_" + ft, xcol]].dropna()
        a.scatter(s[xcol], s["d_" + ft], s=5, alpha=0.3, color=COLORS.get(ft, "#444444"))
        b1, b0 = np.polyfit(s[xcol], s["d_" + ft], 1); xs = np.linspace(s[xcol].min(), s[xcol].max(), 50)
        a.plot(xs, b1 * xs + b0, "k-", lw=1.2); a.axhline(0, color="grey", lw=0.6)
        a.set_xlabel(xlab); a.set_ylabel(f"Δscore ({ft} − base)")
        a.set_title(f"{ft}:  r = {pearsonr(s[xcol], s['d_'+ft])[0]:+.2f}", fontsize=10)
fig.suptitle("Base → fine-tuned score shift by surface charge (Δscore = FT − v_48_002 base)", y=1.0, fontsize=12)
fig.tight_layout(); fig.savefig(os.path.join(DRIVE_RESULTS_DIR, f"ft_score_shift_scatter_{SESSION_ID}.png"), dpi=150)
plt.show()

# ---- Figure 2: correlation summary (the polar comparison in one chart) ----
fig2, ax2 = plt.subplots(figsize=(7, 4.2)); feats = ["charge", "pI", "acidic"]; xpos = np.arange(len(feats)); w = 0.8 / len(present)
for k, ft in enumerate(present):
    s = m[["d_" + ft] + feats].dropna(); rs = [pearsonr(s["d_" + ft], s[f])[0] for f in feats]
    ax2.bar(xpos + (k - (len(present) - 1) / 2) * w, rs, w, color=COLORS.get(ft, "#444444"), label=ft)
ax2.axhline(0, color="k", lw=0.8); ax2.set_xticks(xpos); ax2.set_xticklabels(["net charge", "pI", "acidic frac"])
ax2.set_ylabel("corr(Δscore, property)"); ax2.legend(fontsize=9, frameon=False)
ax2.set_title("Polar score-shift: AlkSecMPNN prefers acidic (neg), AcidSecMPNN basic (pos)")
fig2.tight_layout(); fig2.savefig(os.path.join(DRIVE_RESULTS_DIR, f"ft_score_shift_summary_{SESSION_ID}.png"), dpi=150)
plt.show()

m.to_csv(os.path.join(DRIVE_RESULTS_DIR, f"ft_score_shift_{SESSION_ID}.csv"), index=False)
print("saved ft_score_shift_{scatter,summary}.png + .csv to", DRIVE_RESULTS_DIR)


## ESMIF

In [ ]:
#@title Setup pt 1


!pip install numpy==1.26.4

!pip install torch==2.3.0+cu121 torchvision torchaudio -f https://download.pytorch.org/whl/cu121/torch_stable.html

!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv torch-geometric -f https://data.pyg.org/whl/torch-2.3.0+cu121.html

!pip install biotite==0.39.0

!pip install git+https://github.com/facebookresearch/esm.git

import os
os.kill(os.getpid(), 9)


In [ ]:
# === Google Drive setup for persistent results ===
from google.colab import drive
import os
try:
    drive.mount('/content/drive')
except Exception as _e:
    print(f"Drive mount: {_e}")
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/decoding_bias_results/AF'
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"Results will save to: {DRIVE_RESULTS_DIR}")
# === End Drive setup ===

#@title Setup pt 2
import json
import pandas as pd
import numpy as np
import requests
import torch
import torch.nn.functional as F
import esm
from tqdm import tqdm
from pathlib import Path
import logging
from typing import Optional, Tuple, List
from biotite.sequence.io.fasta import FastaFile, get_sequences
from datetime import datetime
from google.colab import drive
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import esm

# Mount Google Drive
drive.mount('/content/drive')

# Constants
PDB_DIR = "pdb_files"
MODEL_NAME = "esm_if1_gvp4_t16_142M_UR50"
# === Resume-from-checkpoint helpers ===
import time as _t, glob as _g
SESSION_ID = _t.strftime("%Y%m%d_%H%M%S")

def get_completed_entries(model_dir):
    """Return set of Entry IDs already scored (read from any CSV in model_dir)."""
    if not os.path.isdir(model_dir):
        return set()
    done = set()
    for fn in _g.glob(os.path.join(model_dir, "*.csv")):
        try:
            _df = pd.read_csv(fn)
            if "Entry" in _df.columns:
                done.update(_df["Entry"].dropna().unique())
        except Exception as _e:
            print(f"  could not read {fn}: {_e}")
    return done

def csv_filter_resume(input_csv, model_dir):
    """Drop already-scored rows. Returns path to filtered CSV (or original if nothing done)."""
    done = get_completed_entries(model_dir)
    if not done:
        print(f"  Resume: no prior results in {model_dir}; running full CSV")
        return input_csv
    df = pd.read_csv(input_csv)
    before = len(df)
    df = df[~df["Entry"].isin(done)].reset_index(drop=True)
    print(f"  Resume: {len(done)} already done in {model_dir}; "
          f"{len(df)}/{before} remaining (session {SESSION_ID})")
    if len(df) == 0:
        print("  All entries already scored - nothing to do.")
        return None
    out = input_csv.replace(".csv", f"_remaining_{SESSION_ID}.csv")
    df.to_csv(out, index=False)
    return out
# === End resume helpers ===


In [ ]:
#@title Likelihood Predictor

class ESMIFProcessor:
    def __init__(self, output_file: str = "esmif_results.csv", batch_processing: bool = True):
        # Redirect to Drive if a relative path was passed
        if not os.path.isabs(output_file):
            esmif_dir = os.path.join(DRIVE_RESULTS_DIR, "esmif")
            os.makedirs(esmif_dir, exist_ok=True)
            output_file = os.path.join(esmif_dir, f'{SESSION_ID}_{output_file}')
        self.output_file = output_file
        self.batch_processing = batch_processing
        # Use GPU if available
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'

        os.makedirs(PDB_DIR, exist_ok=True)
        self.model, self.alphabet = esm.pretrained.esm_if1_gvp4_t16_142M_UR50()
        self.model = self.model.eval().to(self.device)

    def download_pdb(self, accession: str) -> Tuple[Optional[str], bool]:
        pdb_path = os.path.join(PDB_DIR, f"AF-{accession}-F1-model_v6.pdb")
        url = f"https://alphafold.ebi.ac.uk/files/AF-{accession}-F1-model_v6.pdb"

        if os.path.exists(pdb_path):
            return pdb_path, True

        try:
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                with open(pdb_path, 'wb') as f:
                    f.write(response.content)
                return pdb_path, True
            else:
                logging.warning(f"Failed to download {accession}")
        except requests.RequestException as e:
            logging.error(f"Request failed for {accession}: {e}")

        return None, False

    def score_protein(self, sequence: str, pdb_path: str, chain: str = "A") -> Optional[dict]:
        """
        Score a protein sequence given its structure.
        Returns detailed scoring information including per-position scores and structure validity.
        """
        try:
            # Get backbone coordinates and native sequence
            coords, native_seq = esm.inverse_folding.util.load_coords(pdb_path, chain)

            # Get positional losses comparing to WT sequence
            loss, target_padding_mask = esm.inverse_folding.util.get_sequence_loss(
                self.model, self.alphabet, coords, sequence
            )

            # Calculate sequence scores (log likelihoods)
            ll_fullseq, ll_withcoord = esm.inverse_folding.util.score_sequence(
                self.model, self.alphabet, coords, sequence
            )

            # Calculate valid coordinate mask
            coord_mask = np.all(np.isfinite(coords), axis=(-1, -2))
            valid_positions = np.sum(coord_mask)

            # Calculate valid position scores
            valid_pos_losses = loss[coord_mask]

            return {
                'sequence_length': len(sequence),
                'valid_positions': int(valid_positions),
                'percent_valid': float(valid_positions/len(sequence) * 100),

                # Main scoring metrics
                'total_score': float(ll_fullseq),  # Average over all positions
                'valid_pos_score': float(ll_withcoord),  # Average over valid positions only

                # Per-position score statistics
                'mean_pos_score': float(np.mean(valid_pos_losses)),
                'min_pos_score': float(np.min(valid_pos_losses)),
                'max_pos_score': float(np.max(valid_pos_losses)),
                'std_pos_score': float(np.std(valid_pos_losses)),

                # Structure information
                'has_missing_coords': bool(valid_positions < len(sequence)),
                'num_missing_coords': int(len(sequence) - valid_positions)
            }

        except Exception as e:
            logging.error(f"Error scoring {pdb_path}: {e}")
            return None

    def process_proteins(self, csv_file: str, start_idx: int = 0, end_idx: Optional[int] = None, chain: str = "A") -> pd.DataFrame:
        """Process a range of proteins from the input CSV file."""
        df = pd.read_csv(csv_file)

        if end_idx is None:
            end_idx = len(df)
        df_subset = df.iloc[start_idx:end_idx]

        results = []
        for _, row in tqdm(df_subset.iterrows(), total=len(df_subset),
                          desc=f"Processing proteins {start_idx} to {end_idx-1}"):
            accession = row['Entry']
            sequence = row['sequence']

            # Download structure
            pdb_path, success = self.download_pdb(accession)
            if not success:
                continue

            # Score protein
            score_dict = self.score_protein(sequence, pdb_path, chain)
            if score_dict is not None:
                score_dict['Entry'] = accession  # Add entry ID to results
                results.append(score_dict)

        # Create DataFrame with organized columns
        column_order = [
            'Entry', 'sequence_length', 'valid_positions', 'percent_valid',
            'total_score', 'valid_pos_score',
            'mean_pos_score', 'min_pos_score', 'max_pos_score', 'std_pos_score',
            'has_missing_coords', 'num_missing_coords'
        ]

        results_df = pd.DataFrame(results)
        results_df = results_df[column_order]  # Reorder columns
        return results_df

    def process_all(self, csv_file: str, start_row: int = 0):  # Add start_row parameter
        """Process all proteins, either in batches or all at once, starting from a specified row."""
        if self.batch_processing:
            self.process_in_batches(csv_file, start_row=start_row)  # Pass start_row
        else:
            results_df = self.process_proteins(csv_file, start_idx=start_row)  # Use start_row


            # Print summary statistics
            print("\nProcessing Summary:")
            print(f"Total proteins processed: {len(results_df)}")
            print(f"Mean total score: {results_df['total_score'].mean():.3f}")
            print(f"Mean valid position score: {results_df['valid_pos_score'].mean():.3f}")
            print(f"Average % valid positions: {results_df['percent_valid'].mean():.1f}%")

            print(f"\nResults saved to {self.output_file}")

    def process_in_batches(self, csv_file: str, batch_size: int = 50, start_row: int = 0):  # Add start_row
        """Process proteins in batches, starting from a specified row."""
        df = pd.read_csv(csv_file)
        total_proteins = len(df)
        num_batches = ((total_proteins - start_row) // batch_size) + 1  # Adjust for start_row

        all_results = []
        for batch_idx in range(num_batches):
            start_idx = start_row + batch_idx * batch_size  # Adjust start_idx
            end_idx = min(start_idx + batch_size, total_proteins)

            print(f"\nProcessing batch {batch_idx + 1}/{num_batches}")

            results_df = self.process_proteins(csv_file, start_idx, end_idx)
            all_results.append(results_df)

            # Save intermediate results
            batch_results = pd.concat(all_results)
            batch_results.to_csv(self.output_file, index=False)

            print(f"Processed {len(results_df)} proteins in current batch")
            print(f"Total proteins processed so far: {len(batch_results)}")


In [ ]:
# @title Run
if __name__ == "__main__":
    processor = ESMIFProcessor(batch_processing=True)

    # Option to choose between processing CSV or single sequence
    process_mode = input("Enter mode ('csv' or 'single'): ").strip().lower()

    if process_mode == 'csv':
        # Existing CSV processing logic
        _csv = csv_filter_resume("../data/ACID_output_with_properties.csv", os.path.join(DRIVE_RESULTS_DIR, "esmif"))
        if _csv is not None:
            processor.process_all(_csv, start_row=0)
    elif process_mode == 'single':
        accession = input("Enter AlphaFold PDB accession code (e.g., P0DTD1): ").strip()
        sequence = input("Enter the protein sequence: ").strip().upper()

        if not accession or not sequence:
            print("PDB accession code and sequence are required for single processing.")
        else:
            print(f"Processing single protein: {accession}")
            score_dict = processor.process_single_protein(accession, sequence) # Call the new method

            if score_dict:
                print("\nESMIF Results:")
                for key, value in score_dict.items():
                    print(f"{key}: {value}")
            else:
                print(f"Failed to process protein {accession}.")
    else:
        print("Invalid mode selected. Please enter 'csv' or 'single'.")